In [0]:
%sql
DROP TABLE IF EXISTS la_lakehouse.silver.plum_items

In [0]:
from pyspark.sql.functions import col, trim 
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('la_lakehouse.bronze.plu_agendas')

In [0]:
df.display()

# Extracting Data From Raw Text

In [0]:
df = df.withColumn('raw_text', F.regexp_replace(col('raw_text'), '\u00ad', '-'))

In [0]:
df = df.withColumn('raw_text_split',F.split(col('raw_text'),r"\n\(\d+\)"))

In [0]:
df = df.withColumn('raw_text_split',F.slice(col('raw_text_split'), 2, F.size(col('raw_text_split'))-1))

In [0]:
df = df.select(
    "*", 
    F.posexplode(col('raw_text_split')).alias('item_position', 'item_text')
)

In [0]:

df = df.withColumn('item_number',F.col('item_position') + 1)

In [0]:
df = df.withColumn('item_text', F.trim(col('item_text')))

In [0]:
df = df.withColumn('item_text', F.split(col('item_text'),'GENERAL PUBLIC COMMENT')[0])

In [0]:
df = df.withColumn('council_file_no', F.regexp_extract('item_text', r'(^\d{2}-\d+(?:-\w+)?)', 1))

In [0]:
df = df.withColumn('council_district_raw', F.regexp_extract('item_text', r'CDs? (\d+(?:,\s*\d+)*)', 1))

In [0]:
df = df.withColumn('council_district_split', F.split(col('council_district_raw'), ','))

In [0]:
df = df.withColumn('council_district_split', F.transform(col('council_district_split'), lambda x: F.trim(x)))

In [0]:
df = df.withColumn('council_district_is_multi', F.when(F.size(col('council_district_split')) > 1, True).otherwise(False))
df = df.withColumn('council_district', col('council_district_split')[0])

In [0]:
df = df.withColumn('applicant',F.regexp_extract('item_text',r'\nApplicant: ([^\n]+)',1))

In [0]:
df = df.withColumn('representative', F.regexp_extract('item_text', r'\nRepresentative: ([^\n]+)', 1))

In [0]:
df = df.withColumn('case_no', F.regexp_extract('item_text', r'\nCase No\.?:? ([^\n]+)', 1))

In [0]:
df = df.withColumn('environmental_no', F.regexp_extract('item_text', r'\n(?:Environmental No\.?:?|EIR No\.?:?|Environmental Assessment.*?No\.?:?) ([^\n]+)', 1))

In [0]:
df = df.withColumn('fiscal_impact_statement', F.regexp_extract('item_text', r'\nFiscal Impact Statement: ([^\n]+)', 1))
df = df.withColumn('fiscal_impact_statement', F.regexp_replace(col('fiscal_impact_statement'), r'\.$', ''))

In [0]:
df = df.withColumn('financial_policies_statement', F.regexp_extract('item_text', r'\nFinancial Policies Statement: ([^\n]+)', 1))

In [0]:
df = df.withColumn('continued_from_date',F.regexp_extract('item_text',r'CONTINUED FROM ([^\n]+)',1))

In [0]:
df = df.withColumn('is_continued',F.when(col('item_text').rlike('CONTINUED FROM'),True).otherwise(False))

In [0]:
df = df.withColumn('community_impact_reasons',F.regexp_extract('item_text', r'Community Impact Statement: (.+)', 1))

In [0]:
df = df.withColumn('community_impact_reasons',F.split('community_impact_reasons', r'\nTIME LIMIT FILE')[0])

In [0]:
df = df.withColumn('community_impact_reasons', F.regexp_replace(col('community_impact_reasons'), r'\s+$', ''))

In [0]:
df = df.withColumn('community_impact_submitted',F.when(col('community_impact_reasons').startswith('Yes'),True).otherwise(False))

In [0]:
df = df.withColumn('community_impact_reasons', F.regexp_replace(col('community_impact_reasons'), 'Yes\n', ''))

In [0]:
df = df.withColumn('community_impact_reasons', F.when(col('community_impact_reasons') == 'None submitted', None).otherwise(col('community_impact_reasons')))

#Column Renaming

In [0]:
df = df.withColumnRenamed("templateId", "template_id")
df = df.withColumnRenamed("templateName", "template_name")

#Removing Fragements And Attachment Noise

In [0]:
df = df.filter(col('council_file_no') != '')

#Fixing DataTypes

In [0]:
df = df.withColumn('council_district', F.when(col('council_district') == '', None).otherwise(col('council_district').cast('int')))

df = df.withColumn('meeting_date', F.when(col('meeting_date') == '', None).otherwise(F.to_date(col('meeting_date'), 'MMM dd, yyyy')))

df = df.withColumn('continued_from_date', F.try_to_date(col('continued_from_date'), F.lit('M/d/yy')))


#Testing Dataframe

In [0]:
df.select('item_number').show()

In [0]:
df.select('meeting_date').filter(col('meeting_date').isNull()).count()

In [0]:
df.select('meeting_date').distinct().orderBy('meeting_date').show(5)
df.select('meeting_date').distinct().orderBy(col('meeting_date').desc()).show(5)

In [0]:
df.select('council_district').show()

In [0]:
df.select('applicant').show()

In [0]:
df.select('representative').show()

In [0]:
df.select('case_no').show()

In [0]:
df.select('environmental_no').show()

In [0]:
df.select('fiscal_impact_statement').show()

In [0]:
df.select('continued_from_date').show()

In [0]:
df.select('is_continued').show()

In [0]:
df.orderBy('item_number').select('item_number', 'community_impact_submitted', 'community_impact_reasons').show(truncate=False)

In [0]:
fragment_rows = df.filter(col('council_file_no') == '').select('item_number', 'item_text').limit(10).collect()
for row in fragment_rows:
    print(f"item_number: {row['item_number']}")
    print(row['item_text'][:200])
    print("---")

In [0]:
df.select('item_number', 'council_file_no').show(10, truncate=False)

In [0]:
df.filter(col('document_id') == 84459).select('item_number', 'council_file_no').orderBy('item_number').show(20, truncate=False)

In [0]:
df.filter(col('council_file_no') != '').groupBy('document_id').count().orderBy('count').show(20)

# Writting Dataframe To Silver Catalog

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("la_lakehouse.silver.plum_items")

#Testing Table

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.silver.plum_items

In [0]:
%sql
SELECT * 
FROM la_lakehouse.silver.plum_items
WHERE document_id = 84459;

In [0]:
df.count()

In [0]:
%sql
SELECT YEAR(meeting_date), COUNT(*) 
FROM la_lakehouse.silver.plum_items 
GROUP BY 1 ORDER BY 1

In [0]:
test_text = "21-0934\nCDs 4, 5, 13\nEnvironmental Impact Report (EIR)..."

import re
match = re.search(r'CDs? ([\d, ]+)', test_text)
print(match.group(1) if match else "NO MATCH")

In [0]:
test_text_2 = "26-0622\nCD 5 CONTINUED FROM 6/23/26..."
match_2 = re.search(r'CDs? ([\d, ]+)', test_text_2)
print(match_2.group(1) if match_2 else "NO MATCH")

In [0]:
import re
pattern = r'CDs? (\d+(?:,\s*\d+)*)'

tests = [
    "26-0622\nCD 5 CONTINUED FROM 6/23/26...",
    "21-0934\nCDs 4, 5, 13 Environmental Impact Report...",
    "25-1441\nCD 12 2021-2029 Housing Element Environmental Impact Report...",
    "26-0600-S83\nBudget Recommendation and Department of City Planning report..."
]

for t in tests:
    match = re.search(pattern, t)
    print(match.group(1) if match else "NO MATCH")